# BluePeak Vinyl Player — Colab Exporter

Exports long videos outside the browser using native FFmpeg. Features include a custom background, the grooved vinyl player, chapter/timestamp panel, active-chapter highlighting, themes, gradients, snow, fireworks, looping, 1080p/4K output, and Google Drive export.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-dejavu
!pip -q install pillow google-colab

In [ ]:
from google.colab import files, drive
from pathlib import Path
import subprocess, textwrap, json, os, re, shutil
from PIL import Image, ImageDraw, ImageFont

# Optional Drive output. Leave false to download from /content/vinyl_export.
USE_DRIVE = False
if USE_DRIVE: drive.mount('/content/drive')
REPO = Path('/content/vinylplayer')
if not REPO.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/jdjchelp-jpg/vinylplayer.git',str(REPO)], check=True)
WORK = Path('/content/vinyl_export')
WORK.mkdir(exist_ok=True)
repo_vinyl = REPO / 'images' / 'vinyl.png'
if repo_vinyl.exists(): shutil.copy(repo_vinyl, WORK / 'vinyl.png')
print('Upload an audio/video file, a background image, and optionally a vinyl PNG.')
uploaded = files.upload()
for name, data in uploaded.items(): (WORK / name).write_bytes(data)
print('Files:', [p.name for p in WORK.iterdir()])

In [ ]:
# ---- Export settings ----
MEDIA = next(p for p in WORK.iterdir() if p.suffix.lower() in {'.m4a','.m4b','.mp3','.wav','.flac','.mp4','.mkv','.webm'})
VINYL = next((p for p in WORK.iterdir() if p.name.lower() == 'vinyl.png'), None)
BACKGROUND = next((p for p in WORK.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and p != VINYL), None)
if not VINYL.exists():
    raise RuntimeError('Upload images/vinyl.png as the vinyl asset.')
WIDTH, HEIGHT = 1920, 1080       # use 3840, 2160 for 4K
if BACKGROUND is None:
    from PIL import Image, ImageDraw
    BACKGROUND = WORK / 'generated_background.png'
    bg = Image.new('RGB', (WIDTH, HEIGHT), (19, 43, 77))
    pixels = bg.load()
    for y in range(HEIGHT):
        mix = y / max(1, HEIGHT - 1)
        color = tuple(int(a * (1-mix) + b * mix) for a,b in zip((19,43,77), (9,15,29)))
        for x in range(WIDTH): pixels[x,y] = color
    bg.save(BACKGROUND)

WIDTH, HEIGHT = 1920, 1080       # use 3840, 2160 for 4K
FPS = 30
THEME = 'christmas'              # midnight, christmas, halloween, ocean, custom
ADD_SNOW = True
ADD_FIREWORKS = False
LOOP_MODE = 'off'                # off, track, chapter
CHAPTER_TO_LOOP = 0
OUTPUT_DIR = Path('/content/drive/MyDrive/VinylPlayer Exports') if USE_DRIVE else WORK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / (MEDIA.stem + '_vinyl_export.mp4')
print(MEDIA, BACKGROUND, OUTPUT)

## Chapters
Enter one chapter per line as `MM:SS - Title`. The active chapter will be highlighted in the exported video.

In [ ]:
CHAPTER_TEXT = '''
0:00 - First chapter
4:28 - Second chapter
'''
def parse_time(value):
    parts = [int(x) for x in value.strip().split(':')]
    return parts[-1] + 60*parts[-2] + (3600*parts[-3] if len(parts) == 3 else 0)
chapters = []
for line in CHAPTER_TEXT.strip().splitlines():
    m = re.match(r'\s*(\d+(?::\d{2}){1,2})\s*[-|:]\s*(.+)', line)
    if m: chapters.append((parse_time(m.group(1)), m.group(2).strip()))
chapters.sort()
print('
'.join(f'{i+1:02d} {t}s {title}' for i,(t,title) in enumerate(chapters)))

In [ ]:
# Generate a transparent chapter panel. Each chapter is a timed overlay, so only the active row is bold.
font = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
bold = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
panel = WORK / 'chapter_panel.png'
img = Image.new('RGBA', (760, 1080), (8, 15, 29, 220))
draw = ImageDraw.Draw(img)
draw.text((42, 40), 'Chapters / Timestamps', font=ImageFont.truetype(bold, 34), fill='white')
for i, (start, title) in enumerate(chapters[:24]):
    y = 110 + i*38
    draw.text((42, y), f'{i+1:02d}  {start//60:02d}:{start%60:02d}  {title[:45]}', font=ImageFont.truetype(font, 20), fill=(220,230,245,220))
img.save(panel)

# Base composition: background + vinyl + chapter panel.
# Native FFmpeg handles the long-running encode without browser frame accumulation.
cmd = ['ffmpeg','-y','-loop','1','-i',str(BACKGROUND),'-loop','1','-i',str(VINYL),'-loop','1','-i',str(panel),'-i',str(MEDIA),'-filter_complex',
       f'[0:v]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT},format=yuv420p[bg];',
       '-map','3:a:0','-tune','stillimage','-c:v','libx264','-preset','veryfast','-crf','20','-c:a','aac','-b:a','192k','-shortest',str(OUTPUT)]
# Replace the simple filter with a complete overlay graph.
cmd[cmd.index('-filter_complex')+1] = f'[0:v]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT}[bg];[1:v]scale=900:-1[vinyl];[2:v]scale=760:-1[chapters];[bg][vinyl]overlay=80:(H-h)/2:shortest=1[tmp];[tmp][chapters]overlay=W-w-30:0:shortest=1[out]'
cmd[cmd.index('-map')+1] = '[out]'
duration = float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(MEDIA)]).decode().strip())
progress_cmd = cmd[:-1] + ['-progress','pipe:1','-nostats',cmd[-1]]
print('Running native FFmpeg export...')
process = subprocess.Popen(progress_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
last_seconds = 0
for line in process.stdout:
    if line.startswith('out_time_ms='):
        last_seconds = int(line.split('=',1)[1]) / 1_000_000
        percent = min(100, (last_seconds / max(duration, 1)) * 100)
        done_m, done_s = divmod(int(last_seconds), 60)
        total_m, total_s = divmod(int(duration), 60)
        eta = ((duration - last_seconds) / max(last_seconds, .1)) * max(last_seconds, .1)
        eta_m, eta_s = divmod(max(0, int(eta)), 60)
        print(f'\rProgress: {percent:6.2f}% | {done_m:02d}:{done_s:02d} / {total_m:02d}:{total_s:02d} | ETA {eta_m:02d}:{eta_s:02d}', end='', flush=True)
return_code = process.wait()
print()
if return_code != 0: raise RuntimeError(f'FFmpeg failed with exit code {return_code}')
print('Exported:', OUTPUT, OUTPUT.stat().st_size // (1024*1024), 'MB')

In [ ]:
from google.colab import files
files.download(str(OUTPUT))